# TrIncNet — Plant Disease Classification
**Paper:** TrIncNet: a lightweight vision transformer network for identification of plant diseases  
**Authors:** Pushkar Gole, Punam Bedi, Sudeep Marwaha, Md. Ashraful Haque, Chandan Kumar Deb  
**Journal:** Frontiers in Plant Science, Vol. 14, 2023  
**DOI:** 10.3389/fpls.2023.1221557  

---
## Architecture Summary
- Input: 256x256x3, normalize /255
- Patch size: 16x16 → 256 patches, embed_dim=256
- CLS token + learnable positional embedding (257 tokens total)
- Stack of 2x Trans-Inception blocks (MHA + LayerNorm + Inception module + skip connection)
- Inception: 4 branches concat to 224 channels, then projected to 256 (residual match)
- Head: CLS token [0] → Dense(num_classes, softmax)
- Target: ~6.95M parameters

---
## Experiment Design
- **Scenario 1 (Baseline):** Resize 256x256 + /255 only. No augmentation.
- **Scenario 2 (Augmented):** Rotation + Zoom + Horizontal Flip + Brightness on training set.
- Metrics: Accuracy, F1-Macro, Recall-Macro, Classification Report per class
- Best model checkpoint (val_accuracy) reloaded before test evaluation

## Cell 1: Install Dependencies

In [ ]:
!pip install -q scikit-learn matplotlib numpy openpyxl Pillow

## Cell 2: Imports and Reproducibility Seeds

In [ ]:

import os
import gc
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

# ── GPU Detection ──────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU detected — running on CPU")

# ── Multi-GPU: MirroredStrategy (Rule 8) ──────────────────────────────────────
strategy = tf.distribute.MirroredStrategy()
n_gpus = strategy.num_replicas_in_sync
print(f"[Multi-GPU] MirroredStrategy: {n_gpus} GPU(s) available")


## Cell 3: Configuration

In [ ]:
# ── Dataset Paths ──────────────────────────────────────────────────────────────
# Dataset is pre-split into train/val/test — no splitting needed
DATASET_ROOT = "/kaggle/input/datasets/tminhhi/dataset-ot/dataset_smartcrop_plit_300"
TRAIN_DIR    = os.path.join(DATASET_ROOT, "train")
VAL_DIR      = os.path.join(DATASET_ROOT, "val")
TEST_DIR     = os.path.join(DATASET_ROOT, "test")

# ── Auto-detect class names ────────────────────────────────────────────────────
class_names = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d))
])
NUM_CLASSES = len(class_names)
print(f"Classes detected: {NUM_CLASSES}")
print(f"Class names: {class_names}")

# ── TrIncNet Architecture Hyperparameters (from Table 2, paper) ────────────────
IMAGE_SIZE       = 256          # Paper: 256x256 (Muc 4.2.1, Table 1, Table 2)
PATCH_SIZE       = 16           # Paper: 16x16 (Table 2)
NUM_PATCHES      = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 256 patches
EMBED_DIM        = 256          # Paper: Size of Embedded Patch N=256 (Table 2)
NUM_HEADS        = 12           # Paper: Number of Heads m=12 (Table 2)
KEY_DIM          = EMBED_DIM               # 256 per head — paper param target ~6.95M requires full embed_dim per head, not embed_dim/num_heads
NUM_BLOCKS       = 2            # Paper: Number of Trans-Inception blocks = 2 (Table 2)
LAYER_NORM_EPS   = 1e-6         # Paper: Layer_normalization_rate = 1e-6 (Table 2)

# Inception Module Filters (Table 2: 96, 16, 64, 128, 32, 32)
# Mapping per Appendix Table A1 (GoogLeNet Inception-v1 structure):
# op1: 1x1 direct  = 96
# op2: reduce 1x1  = 16  -> op5: 3x3 = 64
# op3: reduce 1x1  = 128 -> op6: 5x5 = 32
# op7: 1x1 after maxpool = 32
# Total concat channels = 96 + 64 + 32 + 32 = 224
INC_F1   = 96    # branch1: 1x1 direct
INC_F2   = 16    # branch2: reduce 1x1
INC_F5   = 64    # branch2: 3x3 output
INC_F3   = 128   # branch3: reduce 1x1 (NOTE: large reduce; per paper instruction)
INC_F6   = 32    # branch3: 5x5 output
INC_F7   = 32    # branch4: 1x1 after maxpool
INC_OUT  = INC_F1 + INC_F5 + INC_F6 + INC_F7  # = 224
PROJ_DIM = EMBED_DIM  # projection target = 256 (to match residual)

# ── Training Hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE       = 32           # Paper: Muc 4.3
LEARNING_RATE    = 1e-4         # Confirmed by user (paper not stated)
EPOCHS           = 500          # Paper: Muc 4.3
EARLY_STOP_PAT   = 20           # Paper: patience=20 (Muc 4.3)

# ── Output Paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR   = "/kaggle/working"
MODEL_S1     = os.path.join(OUTPUT_DIR, "best_model_s1.keras")
MODEL_S2     = os.path.join(OUTPUT_DIR, "best_model_s2.keras")

# Plot save paths
LC_S1_PATH   = os.path.join(OUTPUT_DIR, "learning_curve_s1.png")
CM_S1_PATH   = os.path.join(OUTPUT_DIR, "confusion_matrix_s1.png")
LC_S2_PATH   = os.path.join(OUTPUT_DIR, "learning_curve_s2.png")
CM_S2_PATH   = os.path.join(OUTPUT_DIR, "confusion_matrix_s2.png")

print(f"\nConfiguration summary:")
print(f"  Image size:   {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Patch size:   {PATCH_SIZE}x{PATCH_SIZE}")
print(f"  Num patches:  {NUM_PATCHES}")
print(f"  Embed dim:    {EMBED_DIM}")
print(f"  Num heads:    {NUM_HEADS}  (key_dim={KEY_DIM} per head)")
print(f"  Num blocks:   {NUM_BLOCKS}")
print(f"  Inception concat channels: {INC_OUT} -> projected to {PROJ_DIM}")
print(f"  Batch size:   {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs:       {EPOCHS} (early stop patience={EARLY_STOP_PAT})")

## Cell 4: Dataset Distribution and Class Imbalance Check

In [ ]:
# ── Count samples per class in train set ──────────────────────────────────────
class_counts = {}
for cls in class_names:
    cls_path = os.path.join(TRAIN_DIR, cls)
    n = len([f for f in os.listdir(cls_path)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))])
    class_counts[cls] = n

total_train = sum(class_counts.values())
max_count   = max(class_counts.values())
min_count   = min(class_counts.values())
imbalance_ratio = max_count / min_count

print(f"Training set distribution ({total_train} total images):")
for cls, n in class_counts.items():
    print(f"  {cls}: {n} ({n/total_train*100:.1f}%)")

print(f"\nMax count: {max_count}  |  Min count: {min_count}")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(max(10, NUM_CLASSES * 0.6), 5))
bars = plt.bar(class_names, [class_counts[c] for c in class_names],
               color='steelblue', edgecolor='white')
plt.title('Dataset Distribution — Training Set', fontsize=14, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right', fontsize=8)
for bar, cls in zip(bars, class_names):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             str(class_counts[cls]), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()
plt.close()

# ── Class weight decision ──────────────────────────────────────────────────────
USE_CLASS_WEIGHTS = (imbalance_ratio >= 1.5)
class_weight_dict = None

if USE_CLASS_WEIGHTS:
    print(f"\n[INFO] Imbalance ratio={imbalance_ratio:.2f} >= 1.5 — computing class weights.")
    # Build label array for sklearn
    y_train_labels = []
    for idx, cls in enumerate(class_names):
        y_train_labels.extend([idx] * class_counts[cls])
    y_train_labels = np.array(y_train_labels)
    cw = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(NUM_CLASSES),
        y=y_train_labels
    )
    class_weight_dict = {i: cw[i] for i in range(NUM_CLASSES)}
    print(f"Class weights: {class_weight_dict}")
else:
    print(f"\n[INFO] Imbalance ratio={imbalance_ratio:.2f} < 1.5 — no class weighting applied.")

## Cell 5: TrIncNet Architecture Builder

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# TrIncNet Architecture (Gole et al., Frontiers in Plant Science 2023)
# ──────────────────────────────────────────────────────────────────────────────

# ── Custom Keras Layers ───────────────────────────────────────────────────────
# All custom layers implement get_config() so they serialize/deserialize
# cleanly from .keras checkpoints without safe_mode issues.

class CLSTokenPrepend(layers.Layer):
    """Prepend a learnable CLS token to a patch embedding sequence."""
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            shape=(1, 1, self.embed_dim),
            initializer='zeros',
            trainable=True,
            name='cls_token'
        )

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls = tf.repeat(self.cls_token, batch_size, axis=0)
        return tf.concat([cls, x], axis=1)

    def get_config(self):
        return {**super().get_config(), 'embed_dim': self.embed_dim}


class AddPositionalEmbedding(layers.Layer):
    """Add a learnable positional embedding to a sequence."""
    def __init__(self, seq_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.embed_dim = embed_dim

    def build(self, input_shape):
        self.pos_embed = self.add_weight(
            shape=(1, self.seq_len, self.embed_dim),
            initializer='zeros',
            trainable=True,
            name='pos_embed'
        )

    def call(self, x):
        return x + self.pos_embed

    def get_config(self):
        return {**super().get_config(), 'seq_len': self.seq_len, 'embed_dim': self.embed_dim}


class SliceCLS(layers.Layer):
    """Extract CLS token: (batch, seq, dim) -> (batch, 1, dim)."""
    def call(self, x):
        return x[:, :1, :]

    def get_config(self):
        return super().get_config()


class SlicePatches(layers.Layer):
    """Extract patch tokens: (batch, seq, dim) -> (batch, seq-1, dim)."""
    def call(self, x):
        return x[:, 1:, :]

    def get_config(self):
        return super().get_config()


class ExtractCLS(layers.Layer):
    """Extract CLS token as vector: (batch, seq, dim) -> (batch, dim)."""
    def call(self, x):
        return x[:, 0, :]

    def get_config(self):
        return super().get_config()


CUSTOM_OBJECTS = {
    'CLSTokenPrepend': CLSTokenPrepend,
    'AddPositionalEmbedding': AddPositionalEmbedding,
    'SliceCLS': SliceCLS,
    'SlicePatches': SlicePatches,
    'ExtractCLS': ExtractCLS,
}


def inception_module(x, f1, f2, f5, f3, f6, f7):
    """
    4-branch Inception module (GoogLeNet Inception-v1 style).
    Input x: (batch, 16, 16, 256).  Concat output = f1+f5+f6+f7 = 224 channels.
    """
    b1 = layers.Conv2D(f1, (1, 1), padding='same', activation='relu')(x)

    b2 = layers.Conv2D(f2, (1, 1), padding='same', activation='relu')(x)
    b2 = layers.Conv2D(f5, (3, 3), padding='same', activation='relu')(b2)

    b3 = layers.Conv2D(f3, (1, 1), padding='same', activation='relu')(x)
    b3 = layers.Conv2D(f6, (5, 5), padding='same', activation='relu')(b3)

    b4 = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    b4 = layers.Conv2D(f7, (1, 1), padding='same', activation='relu')(b4)

    return layers.Concatenate(axis=-1)([b1, b2, b3, b4])


def trans_inception_block(
    x, embed_dim, num_heads, key_dim, layer_norm_eps,
    inc_f1, inc_f2, inc_f5, inc_f3, inc_f6, inc_f7, proj_dim
):
    """
    Trans-Inception Block:
      LayerNorm -> MHA -> residual
      LayerNorm -> separate CLS/patch -> Inception -> Dense proj -> reattach CLS -> residual
    """
    input_block = x

    # Sub-block 1: LayerNorm + MultiHeadAttention
    x1 = layers.LayerNormalization(epsilon=layer_norm_eps)(x)
    x1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=0.0)(x1, x1)
    x1 = layers.Add()([x, x1])

    # Sub-block 2: LayerNorm + Inception
    x2 = layers.LayerNormalization(epsilon=layer_norm_eps)(x1)

    cls_token    = SliceCLS()(x2)      # (batch, 1, embed_dim)
    patch_tokens = SlicePatches()(x2)  # (batch, 256, embed_dim)

    sqrt_p           = IMAGE_SIZE // PATCH_SIZE
    inc_out_channels = inc_f1 + inc_f5 + inc_f6 + inc_f7  # 224

    spatial   = layers.Reshape((sqrt_p, sqrt_p, embed_dim))(patch_tokens)
    spatial   = inception_module(spatial, inc_f1, inc_f2, inc_f5, inc_f3, inc_f6, inc_f7)
    patch_out = layers.Reshape((sqrt_p * sqrt_p, inc_out_channels))(spatial)
    patch_out = layers.Dense(proj_dim, activation='relu')(patch_out)

    cls_out   = layers.Dense(proj_dim, activation='relu')(cls_token)

    x2_out = layers.Concatenate(axis=1)([cls_out, patch_out])

    return layers.Add()([input_block, x2_out])


def build_trincnet(num_classes, image_size, patch_size, embed_dim, num_patches,
                   num_heads, key_dim, num_blocks, layer_norm_eps,
                   inc_f1, inc_f2, inc_f5, inc_f3, inc_f6, inc_f7, proj_dim):
    """Full TrIncNet model."""
    inputs = keras.Input(shape=(image_size, image_size, 3), name='input')

    sqrt_p = image_size // patch_size
    x = layers.Conv2D(
        filters=embed_dim, kernel_size=patch_size, strides=patch_size,
        padding='valid', name='patch_embedding'
    )(inputs)
    x = layers.Reshape((sqrt_p * sqrt_p, embed_dim), name='patch_sequence')(x)

    x = CLSTokenPrepend(embed_dim, name='cls_token_prepend')(x)
    x = AddPositionalEmbedding(num_patches + 1, embed_dim, name='pos_embedding')(x)

    for _ in range(num_blocks):
        x = trans_inception_block(
            x, embed_dim=embed_dim, num_heads=num_heads, key_dim=key_dim,
            layer_norm_eps=layer_norm_eps,
            inc_f1=inc_f1, inc_f2=inc_f2, inc_f5=inc_f5,
            inc_f3=inc_f3, inc_f6=inc_f6, inc_f7=inc_f7,
            proj_dim=proj_dim
        )

    cls_output = ExtractCLS(name='extract_cls')(x)
    outputs    = layers.Dense(num_classes, activation='softmax', name='classifier')(cls_output)

    return Model(inputs=inputs, outputs=outputs, name='TrIncNet')


print("TrIncNet architecture builder defined.")
print(f"Inception concat channels: {INC_OUT}  (96+64+32+32)")
print(f"Projection to: {PROJ_DIM}  (embed_dim, for residual compatibility)")


## Cell 6: Utility Functions (Evaluation, Visualization)

In [ ]:
def evaluate_model(model, test_generator, class_names):
    """
    Run inference on test generator, return metrics dict.
    Returns:
        metrics dict with: accuracy, f1_macro, recall_macro, total_params, model_name
        y_true, y_pred arrays
    """
    test_generator.reset()
    steps = len(test_generator)
    y_pred_probs = model.predict(test_generator, steps=steps, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)

    y_true = test_generator.classes[:len(y_pred)]

    acc        = accuracy_score(y_true, y_pred)
    f1_macro   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    total_p    = model.count_params()

    metrics = {
        'model_name':    'TrIncNet',
        'accuracy':      acc,
        'f1_macro':      f1_macro,
        'recall_macro':  rec_macro,
        'total_params':  total_p
    }
    return metrics, y_true, y_pred


def plot_confusion_matrix(y_true, y_pred, class_names, save_path, title='Confusion Matrix'):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    n_classes = len(class_names)
    fig_size = max(8, n_classes * 0.6)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)

    tick_marks = np.arange(n_classes)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_names, fontsize=7)

    fmt = 'd'
    thresh = cm.max() / 2.0
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha='center', va='center', fontsize=6,
                    color='white' if cm[i, j] > thresh else 'black')

    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Confusion matrix saved: {save_path}")


def plot_learning_curve(history, save_path, title='Learning Curve'):
    """Plot and save training/validation accuracy and loss curves."""
    epochs_ran = len(history.history['accuracy'])
    epoch_range = range(1, epochs_ran + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(epoch_range, history.history['accuracy'],   label='Train Acc', color='royalblue')
    axes[0].plot(epoch_range, history.history['val_accuracy'], label='Val Acc',   color='tomato', linestyle='--')
    axes[0].set_title('Accuracy', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(epoch_range, history.history['loss'],     label='Train Loss', color='royalblue')
    axes[1].plot(epoch_range, history.history['val_loss'], label='Val Loss',   color='tomato', linestyle='--')
    axes[1].set_title('Loss', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Learning curve saved: {save_path}")


print("Utility functions defined.")

---
# SCENARIO 1 — BASELINE (No Augmentation)
Training transform: Resize 256x256 + /255 only

## Cell 7: Scenario 1 — Data Generators (Baseline)

In [ ]:
# ── Scenario 1: Baseline generators ───────────────────────────────────────────
# Normalization: /255 (paper Section 4.2.2 — NOT ImageNet mean/std)
# Training: ONLY resize + /255, no augmentation
# Val / Test: same transform as training baseline

baseline_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen_s1 = baseline_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen_s1 = baseline_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen_s1 = baseline_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Update class_names from generator (authoritative order)
class_names = list(train_gen_s1.class_indices.keys())

print(f"Train samples:      {train_gen_s1.samples}")
print(f"Validation samples: {val_gen_s1.samples}")
print(f"Test samples:       {test_gen_s1.samples}")
print(f"Class index map:    {train_gen_s1.class_indices}")
print(f"Num classes:        {len(class_names)}")

## Cell 8: Scenario 1 — Build and Summarize Model

In [ ]:

# ── Build TrIncNet model for Scenario 1 (inside strategy.scope() for multi-GPU) ─
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

with strategy.scope():
    model_s1 = build_trincnet(
        num_classes     = NUM_CLASSES,
        image_size      = IMAGE_SIZE,
        patch_size      = PATCH_SIZE,
        embed_dim       = EMBED_DIM,
        num_patches     = NUM_PATCHES,
        num_heads       = NUM_HEADS,
        key_dim         = KEY_DIM,
        num_blocks      = NUM_BLOCKS,
        layer_norm_eps  = LAYER_NORM_EPS,
        inc_f1=INC_F1, inc_f2=INC_F2, inc_f5=INC_F5,
        inc_f3=INC_F3, inc_f6=INC_F6, inc_f7=INC_F7,
        proj_dim=PROJ_DIM
    )

    model_s1.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=CategoricalCrossentropy(),
        metrics=['accuracy']
    )

# Model summary (CORE RULE #5)
model_s1.summary(expand_nested=False)

total_params = model_s1.count_params()
print(f"\nTotal parameters: {total_params:,}  ({total_params/1e6:.2f}M)")
print(f"Paper target: ~6.95M")
paper_target = 6.95e6
deviation_pct = abs(total_params - paper_target) / paper_target * 100
print(f"Deviation from paper: {deviation_pct:.1f}% (acceptable if < 10%)")


## Cell 9: Scenario 1 — Training

In [ ]:

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks_s1 = [
    ModelCheckpoint(
        filepath=MODEL_S1,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=EARLY_STOP_PAT,
        restore_best_weights=False,  # we reload from checkpoint below
        mode='max',
        verbose=1
    )
]

# ── Train Scenario 1 ──────────────────────────────────────────────────────────
print("=" * 60)
print("SCENARIO 1 — Baseline (No Augmentation)")
print("Training: Resize 256x256 + /255 ONLY")
print("=" * 60)

steps_per_epoch_s1   = len(train_gen_s1)
validation_steps_s1  = len(val_gen_s1)

history_s1 = model_s1.fit(
    train_gen_s1,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch_s1,
    validation_data=val_gen_s1,
    validation_steps=validation_steps_s1,
    callbacks=callbacks_s1,
    class_weight=class_weight_dict,
    verbose=1
)

print("\nScenario 1 training complete.")
print(f"Best model saved to: {MODEL_S1}")


## Cell 10: Scenario 1 — Learning Curve

In [ ]:
plot_learning_curve(
    history_s1,
    save_path=LC_S1_PATH,
    title='TrIncNet — Scenario 1 (Baseline) Learning Curve'
)
BASELINE_LEARNING_CURVE_PATH = LC_S1_PATH

## Cell 11: Scenario 1 — Classification Report

In [ ]:
# ── Reload best checkpoint before test evaluation (CORE RULE #10) ─────────────
print(f"Loading best model from: {MODEL_S1}")
model_s1 = keras.models.load_model(MODEL_S1, compile=False, custom_objects=CUSTOM_OBJECTS)
model_s1.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=CategoricalCrossentropy(),
    metrics=['accuracy']
)

baseline_metrics, y_true_s1, y_pred_s1 = evaluate_model(
    model_s1, test_gen_s1, class_names
)

print("\n" + "=" * 60)
print("SCENARIO 1 — TEST SET RESULTS (Best Checkpoint)")
print("=" * 60)
print(f"  Accuracy:      {baseline_metrics['accuracy']:.4f}  ({baseline_metrics['accuracy']*100:.2f}%)")
print(f"  F1-Macro:      {baseline_metrics['f1_macro']:.4f}")
print(f"  Recall-Macro:  {baseline_metrics['recall_macro']:.4f}")
print(f"  Total Params:  {baseline_metrics['total_params']:,}  ({baseline_metrics['total_params']/1e6:.2f}M)")

print("\n--- Classification Report (per class) ---")
print(classification_report(
    y_true_s1, y_pred_s1,
    target_names=class_names,
    digits=4,
    zero_division=0
))

## Cell 12: Scenario 1 — Confusion Matrix

In [ ]:
plot_confusion_matrix(
    y_true_s1, y_pred_s1,
    class_names=class_names,
    save_path=CM_S1_PATH,
    title='TrIncNet — Scenario 1 (Baseline) Confusion Matrix'
)
BASELINE_CONFUSION_MATRIX_PATH = CM_S1_PATH

## Cell 13: Clear Session After Scenario 1

In [ ]:
# ── Clear GPU memory before Scenario 2 (CORE RULE #13) ───────────────────────
del model_s1
tf.keras.backend.clear_session()
gc.collect()
print("Session cleared. Ready for Scenario 2.")

---
# SCENARIO 2 — AUGMENTED
Training: Rotation + Zoom + Horizontal Flip + Brightness  
Val/Test: same as Scenario 1 baseline (no augmentation)

## Cell 14: Scenario 2 — Data Generators (Augmented)

In [ ]:
# ── Scenario 2: Augmented training generator ───────────────────────────────────
# 4 mandatory augmentation techniques (CORE RULE #3):
#   1. Rotation (rotation_range=30)
#   2. Zoom/Scale (zoom_range=0.2)
#   3. Horizontal Flip (horizontal_flip=True)
#   4. Brightness (brightness_range=[0.8, 1.2])
# Val/Test: no augmentation (same as Scenario 1 baseline)

aug_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)  # no aug for val/test

train_gen_s2 = aug_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen_s2 = eval_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen_s2 = eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"Train samples (Scenario 2): {train_gen_s2.samples}")
print(f"Val samples:                {val_gen_s2.samples}")
print(f"Test samples:               {test_gen_s2.samples}")
print(f"Augmentation: rotation=30, zoom=0.2, h_flip=True, brightness=[0.8, 1.2]")

## Cell 15: Scenario 2 — Build and Summarize Model

In [ ]:

# ── Reinitialize model from scratch for Scenario 2 (inside strategy.scope()) ──
tf.random.set_seed(SEED)

with strategy.scope():
    model_s2 = build_trincnet(
        num_classes     = NUM_CLASSES,
        image_size      = IMAGE_SIZE,
        patch_size      = PATCH_SIZE,
        embed_dim       = EMBED_DIM,
        num_patches     = NUM_PATCHES,
        num_heads       = NUM_HEADS,
        key_dim         = KEY_DIM,
        num_blocks      = NUM_BLOCKS,
        layer_norm_eps  = LAYER_NORM_EPS,
        inc_f1=INC_F1, inc_f2=INC_F2, inc_f5=INC_F5,
        inc_f3=INC_F3, inc_f6=INC_F6, inc_f7=INC_F7,
        proj_dim=PROJ_DIM
    )

    model_s2.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=CategoricalCrossentropy(),
        metrics=['accuracy']
    )

model_s2.summary(expand_nested=False)
print(f"\nTotal parameters (S2): {model_s2.count_params():,}  ({model_s2.count_params()/1e6:.2f}M)")


## Cell 16: Scenario 2 — Training

In [ ]:

# ── Callbacks for Scenario 2 ──────────────────────────────────────────────────
callbacks_s2 = [
    ModelCheckpoint(
        filepath=MODEL_S2,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=EARLY_STOP_PAT,
        restore_best_weights=False,
        mode='max',
        verbose=1
    )
]

# ── Train Scenario 2 ──────────────────────────────────────────────────────────
print("=" * 60)
print("SCENARIO 2 — Augmented")
print("Augmentation: Rotation=30, Zoom=0.2, H-Flip, Brightness=[0.8,1.2]")
print("=" * 60)

steps_per_epoch_s2   = len(train_gen_s2)
validation_steps_s2  = len(val_gen_s2)

history_s2 = model_s2.fit(
    train_gen_s2,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch_s2,
    validation_data=val_gen_s2,
    validation_steps=validation_steps_s2,
    callbacks=callbacks_s2,
    class_weight=class_weight_dict,
    verbose=1
)

print("\nScenario 2 training complete.")
print(f"Best model saved to: {MODEL_S2}")


## Cell 17: Scenario 2 — Learning Curve

In [ ]:
plot_learning_curve(
    history_s2,
    save_path=LC_S2_PATH,
    title='TrIncNet — Scenario 2 (Augmented) Learning Curve'
)
AUG_LEARNING_CURVE_PATH = LC_S2_PATH

## Cell 18: Scenario 2 — Classification Report

In [ ]:
# ── Reload best checkpoint before test evaluation (CORE RULE #10) ─────────────
print(f"Loading best model from: {MODEL_S2}")
model_s2 = keras.models.load_model(MODEL_S2, compile=False, custom_objects=CUSTOM_OBJECTS)
model_s2.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=CategoricalCrossentropy(),
    metrics=['accuracy']
)

aug_metrics, y_true_s2, y_pred_s2 = evaluate_model(
    model_s2, test_gen_s2, class_names
)

print("\n" + "=" * 60)
print("SCENARIO 2 — TEST SET RESULTS (Best Checkpoint)")
print("=" * 60)
print(f"  Accuracy:      {aug_metrics['accuracy']:.4f}  ({aug_metrics['accuracy']*100:.2f}%)")
print(f"  F1-Macro:      {aug_metrics['f1_macro']:.4f}")
print(f"  Recall-Macro:  {aug_metrics['recall_macro']:.4f}")
print(f"  Total Params:  {aug_metrics['total_params']:,}  ({aug_metrics['total_params']/1e6:.2f}M)")

print("\n--- Classification Report (per class) ---")
print(classification_report(
    y_true_s2, y_pred_s2,
    target_names=class_names,
    digits=4,
    zero_division=0
))

## Cell 19: Scenario 2 — Confusion Matrix

In [ ]:
plot_confusion_matrix(
    y_true_s2, y_pred_s2,
    class_names=class_names,
    save_path=CM_S2_PATH,
    title='TrIncNet — Scenario 2 (Augmented) Confusion Matrix'
)
AUG_CONFUSION_MATRIX_PATH = CM_S2_PATH

## Cell 20: Clear Session After Scenario 2

In [ ]:
del model_s2
tf.keras.backend.clear_session()
gc.collect()
print("Session cleared after Scenario 2.")

---
# RESULTS SUMMARY

## Cell 21: Results Comparison

In [ ]:
print("\n" + "=" * 70)
print("TRINCNET — FINAL RESULTS COMPARISON")
print("=" * 70)
print(f"{'Metric':<20} {'Scenario 1 (Baseline)':>22} {'Scenario 2 (Augmented)':>22}")
print("-" * 70)
print(f"{'Accuracy':<20} {baseline_metrics['accuracy']*100:>21.2f}% {aug_metrics['accuracy']*100:>21.2f}%")
print(f"{'F1-Macro':<20} {baseline_metrics['f1_macro']:>22.4f} {aug_metrics['f1_macro']:>22.4f}")
print(f"{'Recall-Macro':<20} {baseline_metrics['recall_macro']:>22.4f} {aug_metrics['recall_macro']:>22.4f}")
print(f"{'Total Params':<20} {baseline_metrics['total_params']/1e6:>21.2f}M {aug_metrics['total_params']/1e6:>21.2f}M")
print("=" * 70)
print(f"\nPaper target (PlantVillage): Accuracy=99.93%, F1=99.91%, Recall=99.91%, Params=6.95M")

## Cell 22: Install Excel Export Dependencies

In [ ]:
!pip install -q openpyxl Pillow

## Cell 23: Export results_summary.xlsx

In [ ]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "results_summary.xlsx")
MODEL_NAME        = "TrIncNet"

HEADERS    = ["STT", "Ten Model", "Accuracy", "F1-score", "Recall",
              "Learning Curve", "Confusion Matrix", "Parameters"]
COL_WIDTHS = [6, 35, 12, 12, 12, 38, 38, 18]
IMG_ROW_HEIGHT = 200


def style_header_row(ws):
    fill   = PatternFill("solid", fgColor="1F4E79")
    font   = Font(bold=True, color="FFFFFF", size=11)
    border = Border(
        bottom=Side(style="medium", color="FFFFFF"),
        right=Side(style="thin", color="FFFFFF"),
    )
    for col, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill, cell.font, cell.border = fill, font, border
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)


def write_result_row(ws, row_num, stt, metrics, lc_path, cm_path):
    data = [
        stt,
        metrics["model_name"],
        round(metrics["accuracy"], 4),
        round(metrics["f1_macro"], 4),
        round(metrics["recall_macro"], 4),
        "",
        "",
        f"{metrics['total_params']:,}",
    ]
    for col, val in enumerate(data, start=1):
        cell = ws.cell(row=row_num, column=col, value=val)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.row_dimensions[row_num].height = IMG_ROW_HEIGHT

    if os.path.exists(lc_path):
        img_lc = XLImage(lc_path)
        img_lc.width, img_lc.height = 280, 190
        ws.add_image(img_lc, f"F{row_num}")

    if os.path.exists(cm_path):
        img_cm = XLImage(cm_path)
        img_cm.width, img_cm.height = 280, 190
        ws.add_image(img_cm, f"G{row_num}")


def build_sheet(wb, sheet_name, metrics, lc_path, cm_path):
    ws = wb.create_sheet(title=sheet_name)
    for col, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col)].width = width
    ws.row_dimensions[1].height = 30
    style_header_row(ws)
    write_result_row(ws, row_num=2, stt=1, metrics=metrics,
                     lc_path=lc_path, cm_path=cm_path)
    return ws


wb = Workbook()
wb.remove(wb.active)

build_sheet(
    wb, "Baseline",
    metrics=baseline_metrics,
    lc_path=BASELINE_LEARNING_CURVE_PATH,
    cm_path=BASELINE_CONFUSION_MATRIX_PATH
)

build_sheet(
    wb, "Augmented",
    metrics=aug_metrics,
    lc_path=AUG_LEARNING_CURVE_PATH,
    cm_path=AUG_CONFUSION_MATRIX_PATH
)

wb.save(EXCEL_OUTPUT_PATH)
print(f"[Excel] Saved: {EXCEL_OUTPUT_PATH}")
print(f"  Baseline  -> Accuracy={baseline_metrics['accuracy']:.4f} | F1={baseline_metrics['f1_macro']:.4f} | Recall={baseline_metrics['recall_macro']:.4f} | Params={baseline_metrics['total_params']:,}")
print(f"  Augmented -> Accuracy={aug_metrics['accuracy']:.4f}      | F1={aug_metrics['f1_macro']:.4f}      | Recall={aug_metrics['recall_macro']:.4f}      | Params={aug_metrics['total_params']:,}")